In [1]:
import pandas as pd

## BOOKS

In [2]:
# Extract columns from Books

In [3]:
books_df = pd.read_csv(r'..\_rouse_extracts\_originals\Books.csv')
books_df.head()

,osisName,bookOrder,bookName,bookDiv,testament,shortName,slug,yearWritten,placeWritten,chapters,chapterCount,verseCount,writers,peopleCount,placeCount,modified
0,Gen,1,Genesis,Pentateuch,Old Testament,Ge,gen,NaN,NaN,"Gen.1,Gen.2,Gen.3,Gen.4,Gen.5,Gen.6,Gen.7,Gen....",50,1533,moses_2108,1082,266,NaN
1,Exod,2,Exodus,Pentateuch,Old Testament,Ex,exod,NaN,NaN,"Exod.1,Exod.2,Exod.3,Exod.4,Exod.5,Exod.6,Exod...",40,1213,moses_2108,619,179,NaN
2,Lev,3,Leviticus,Pentateuch,Old Testament,Le,lev,NaN,NaN,"Lev.1,Lev.2,Lev.3,Lev.4,Lev.5,Lev.6,Lev.7,Lev....",27,859,moses_2108,366,28,NaN
3,Num,4,Numbers,Pentateuch,Old Testament,Nu,num,NaN,NaN,"Num.1,Num.2,Num.3,Num.4,Num.5,Num.6,Num.7,Num....",36,1288,moses_2108,832,207,NaN
4,Deut,5,Deuteronomy,Pentateuch,Old Testament,Dt,deut,NaN,NaN,"Deut.1,Deut.2,Deut.3,Deut.4,Deut.5,Deut.6,Deut...",34,959,moses_2108,528,146,NaN


In [4]:
for i, col in enumerate(books_df.columns):
    print(f"{i}: {col}")

0: osisName
1: bookOrder
2: bookName
3: bookDiv
4: testament
5: shortName
6: slug
7: yearWritten
8: placeWritten
9: chapters
10: chapterCount
11: verseCount
12: writers
13: peopleCount
14: placeCount
15: modified


In [5]:
books_erd_df = pd.read_csv(r'..\_rouse_extracts\_originals\Books.csv', usecols=['bookOrder', 'shortName', 'osisName', 'bookName', 'bookDiv', 'testament'])
books_erd_df = books_erd_df.loc[:, ['bookOrder', 'shortName', 'osisName', 'bookName', 'bookDiv', 'testament']]
books_erd_df

,bookOrder,shortName,osisName,bookName,bookDiv,testament
0,1,Ge,Gen,Genesis,Pentateuch,Old Testament
1,2,Ex,Exod,Exodus,Pentateuch,Old Testament
2,3,Le,Lev,Leviticus,Pentateuch,Old Testament
3,4,Nu,Num,Numbers,Pentateuch,Old Testament
4,5,Dt,Deut,Deuteronomy,Pentateuch,Old Testament
...,...,...,...,...,...,...
61,62,1Jn,1John,1 John,General Epistles,New Testament
62,63,2Jn,2John,2 John,General Epistles,New Testament
63,64,3Jn,3John,3 John,General Epistles,New Testament
64,65,Jud,Jude,Jude,General Epistles,New Testament


In [6]:
books_erd_df.to_csv(r'..\acts_and_miracles\DIM_books.csv', index=False)

### Expand Writers column

In [7]:
book_writers_df = pd.read_csv(r'..\_rouse_extracts\_originals\Books.csv', usecols=['bookOrder', 'osisName', 'writers'])

In [8]:
book_writers_df['numWriters'] = book_writers_df['writers'].fillna('').str.split(',').apply(lambda x: len(set(item.strip() for item in x if item.strip() != '')))
book_writers_df

,osisName,bookOrder,writers,numWriters
0,Gen,1,moses_2108,1
1,Exod,2,moses_2108,1
2,Lev,3,moses_2108,1
3,Num,4,moses_2108,1
4,Deut,5,moses_2108,1
...,...,...,...,...
61,1John,62,john_1677,1
62,2John,63,john_1677,1
63,3John,64,john_1677,1
64,Jude,65,jude_1756,1


In [9]:
book_writers_df[book_writers_df.numWriters>1]

,osisName,bookOrder,writers,numWriters
8,1Sam,9,"samuel_2469, samuel_2469, gad_1263, nathan_2153",3
9,2Sam,10,"samuel_2469, gad_1263, nathan_2153",3
18,Ps,19,", david_994, asaph_330, solomon_2762, heman_14...",6
19,Prov,20,"solomon_2762, agur_112, lemuel_1817",3


In [10]:
# Explode writers column so each writer gets its own row
book_writers_df = book_writers_df.assign(writers=book_writers_df['writers'].str.split(',')).explode('writers')
book_writers_df['writer'] = book_writers_df['writers'].str.strip()
#books_to_writers_df

In [11]:
#book_writers_df[book_writers_df.numWriters>1]

In [12]:
# Remove duplicates and rows with empty values for writer
book_writers_df['writers'] = book_writers_df['writers'].str.replace(r'\s+', ' ', regex=True).str.strip()
book_writers_df = book_writers_df[book_writers_df['writers'] != ''].drop_duplicates()
#book_writers_df[book_writers_df.numWriters>1].reset_index()

In [13]:
# Select columns
book_writers_erd_df = book_writers_df.iloc[:, [1,4]]
#book_writers_erd_df

In [18]:
#book_writers_erd_df[book_writers_erd_df['bookOrder'].isin(book_writers_erd_df['bookOrder'].value_counts()[book_writers_erd_df['bookOrder'].value_counts() > 1].index)]

In [16]:
# Export to file
book_writers_erd_df.to_csv(r'..\acts_and_miracles\DIM_book_writers.csv', index=False)

## CHAPTERS

In [ ]:
chapters_df = pd.read_csv(r'..\_rouse_extracts\_originals\Chapters.csv')
chapters_df.head().sort_values(by=['book', 'chapterNum'])

In [ ]:
for i, col in enumerate(chapters_df.columns):
    print(f"{i}: {col}")

In [ ]:
#chapters_df['osisRef']=chapters_df['osisRef'].apply(lambda x: x.replace('.', '') if x else x)

### Expand Writers column

In [52]:
chapter_writers_df = pd.read_csv(r'..\_rouse_extracts\_originals\Chapters.csv', usecols=['osisRef', 'book', 'chapterNum', 'writer', 'writer count'])
chapter_writers_df.columns = ['osisRef', 'book', 'chapterNum', 'writers', 'numWriters']
chapter_writers_df = chapter_writers_df.loc[:, ['osisRef', 'book', 'chapterNum', 'writers', 'numWriters']].sort_values(by=['book', 'chapterNum']).reset_index(drop=True)
chapter_writers_df

,osisRef,book,chapterNum,writers,numWriters
0,1Chr.1,1Chr,1,ezra_1245,1
1,1Chr.2,1Chr,2,ezra_1245,1
2,1Chr.3,1Chr,3,ezra_1245,1
3,1Chr.4,1Chr,4,ezra_1245,1
4,1Chr.5,1Chr,5,ezra_1245,1
...,...,...,...,...,...
1184,Zech.13,Zech,13,zechariah_3016,1
1185,Zech.14,Zech,14,zechariah_3016,1
1186,Zeph.1,Zeph,1,zephaniah_3040,1
1187,Zeph.2,Zeph,2,zephaniah_3040,1


In [46]:
#chapter_writers_df[chapter_writers_df.numWriters>1].sort_values(by=['book', 'chapterNum'])

In [53]:
# Explode writers column so each writer gets its own row
chapter_writers_df = chapter_writers_df.assign(writers=chapter_writers_df['writers'].str.split(',')).explode('writers')
chapter_writers_df['writer'] = chapter_writers_df['writers'].str.strip()
chapter_writers_df

,osisRef,book,chapterNum,writers,numWriters,writer
0,1Chr.1,1Chr,1,ezra_1245,1,ezra_1245
1,1Chr.2,1Chr,2,ezra_1245,1,ezra_1245
2,1Chr.3,1Chr,3,ezra_1245,1,ezra_1245
3,1Chr.4,1Chr,4,ezra_1245,1,ezra_1245
4,1Chr.5,1Chr,5,ezra_1245,1,ezra_1245
...,...,...,...,...,...,...
1184,Zech.13,Zech,13,zechariah_3016,1,zechariah_3016
1185,Zech.14,Zech,14,zechariah_3016,1,zechariah_3016
1186,Zeph.1,Zeph,1,zephaniah_3040,1,zephaniah_3040
1187,Zeph.2,Zeph,2,zephaniah_3040,1,zephaniah_3040


In [55]:
chapter_writers_df[chapter_writers_df.numWriters>1]

,osisRef,book,chapterNum,writers,numWriters,writer
102,1Sam.26,1Sam,26,samuel_2469,3,samuel_2469
102,1Sam.26,1Sam,26,gad_1263,3,gad_1263
102,1Sam.26,1Sam,26,nathan_2153,3,nathan_2153
103,1Sam.27,1Sam,27,samuel_2469,3,samuel_2469
103,1Sam.27,1Sam,27,gad_1263,3,gad_1263
...,...,...,...,...,...,...
219,2Sam.23,2Sam,23,gad_1263,3,gad_1263
219,2Sam.23,2Sam,23,nathan_2153,3,nathan_2153
220,2Sam.24,2Sam,24,samuel_2469,3,samuel_2469
220,2Sam.24,2Sam,24,gad_1263,3,gad_1263


In [57]:
# Remove duplicates and rows with empty values for writer
chapter_writers_df['writers'] = chapter_writers_df['writers'].str.replace(r'\s+', ' ', regex=True).str.strip()
chapter_writers_df = chapter_writers_df[chapter_writers_df['writers'] != ''].drop_duplicates()
chapter_writers_df

,osisRef,book,chapterNum,writers,numWriters,writer
102,1Sam.26,1Sam,26,samuel_2469,3,samuel_2469
102,1Sam.26,1Sam,26,gad_1263,3,gad_1263
102,1Sam.26,1Sam,26,nathan_2153,3,nathan_2153
103,1Sam.27,1Sam,27,samuel_2469,3,samuel_2469
103,1Sam.27,1Sam,27,gad_1263,3,gad_1263
...,...,...,...,...,...,...
219,2Sam.23,2Sam,23,gad_1263,3,gad_1263
219,2Sam.23,2Sam,23,nathan_2153,3,nathan_2153
220,2Sam.24,2Sam,24,samuel_2469,3,samuel_2469
220,2Sam.24,2Sam,24,gad_1263,3,gad_1263


In [61]:
chapter_writers_df[chapter_writers_df.numWriters>1]

,osisRef,book,chapterNum,writers,numWriters,writer
102,1Sam.26,1Sam,26,samuel_2469,3,samuel_2469
102,1Sam.26,1Sam,26,gad_1263,3,gad_1263
102,1Sam.26,1Sam,26,nathan_2153,3,nathan_2153
103,1Sam.27,1Sam,27,samuel_2469,3,samuel_2469
103,1Sam.27,1Sam,27,gad_1263,3,gad_1263
...,...,...,...,...,...,...
219,2Sam.23,2Sam,23,gad_1263,3,gad_1263
219,2Sam.23,2Sam,23,nathan_2153,3,nathan_2153
220,2Sam.24,2Sam,24,samuel_2469,3,samuel_2469
220,2Sam.24,2Sam,24,gad_1263,3,gad_1263


In [63]:
# Select columns
chapter_writers_erd_df = chapter_writers_df.iloc[:, [0,1,2,5]].reset_index(drop=True)
chapter_writers_erd_df

,osisRef,book,chapterNum,writer
0,1Chr.1,1Chr,1,ezra_1245
1,1Chr.2,1Chr,2,ezra_1245
2,1Chr.3,1Chr,3,ezra_1245
3,1Chr.4,1Chr,4,ezra_1245
4,1Chr.5,1Chr,5,ezra_1245
...,...,...,...,...
1244,Zech.13,Zech,13,zechariah_3016
1245,Zech.14,Zech,14,zechariah_3016
1246,Zeph.1,Zeph,1,zephaniah_3040
1247,Zeph.2,Zeph,2,zephaniah_3040


In [66]:
# Export to file
chapter_writers_erd_df.to_csv(r'..\acts_and_miracles\DIM_chapter_writers.csv', index_label='BookWriterID')

## VERSES

In [69]:
verses_df = pd.read_csv(r'..\_rouse_extracts\_originals\Verses.csv')
verses_df.head()

,osisRef,status,verseID,book,chapter,verseNum,verseText,richText,mdText,people,peopleCount,places,placesCount,yearNum,peopleGroups,eventsDescribed,timeline,modified
0,Gen.1.1,publish,1001001,Gen,Gen.1,1,In the beginning God created the heaven and th...,In the beginning God created the heaven and th...,In the beginning [God]([/person/god_1324) crea...,god_1324,1,NaN,0,-4004.0,NaN,God creates all things,Creation of all things,2021-01-08 15:07
1,Gen.1.2,publish,1001002,Gen,Gen.1,2,"And the earth was without form, and void; and ...","And the earth was without form, and void; and ...","And the earth was without form, and void; and ...","god_1324,holy_spirit_7400",2,NaN,0,-4004.0,NaN,God creates all things,Creation of all things,2021-01-08 15:07
2,Gen.1.3,publish,1001003,Gen,Gen.1,3,"And God said, Let there be light: and there wa...","And God said, Let there be light: and there wa...","And [God]([/person/god_1324) said, Let there b...",god_1324,1,NaN,0,-4004.0,NaN,God creates all things,Creation of all things,2021-01-08 15:07
3,Gen.1.4,publish,1001004,Gen,Gen.1,4,"And God saw the light, that it was good: and G...","And God saw the light, that it was good: and G...","And [God]([/person/god_1324) saw the light, th...",god_1324,1,NaN,0,-4004.0,NaN,God creates all things,Creation of all things,2021-01-08 15:07
4,Gen.1.5,publish,1001005,Gen,Gen.1,5,"And God called the light Day, and the darkness...","And God called the light Day, and the darkness...",And [God]([/person/god_1324) called the light ...,god_1324,1,NaN,0,-4004.0,NaN,God creates all things,Creation of all things,2021-01-08 15:07


In [70]:
for i, col in enumerate(verses_df.columns):
    print(f"{i}: {col}")

0: osisRef
1: status
2: verseID
3: book
4: chapter
5: verseNum
6: verseText
7: richText
8: mdText
9: people
10: peopleCount
11: places
12: placesCount
13: yearNum
14: peopleGroups
15: eventsDescribed
16: timeline
17: modified
